# Week 5 : Assignment

In [37]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing? 

### Ans: 
The main limitations that make traditional MapReduce not preferred choice and Spark a preferred choice can be following: 
1. Because of excessive I/O operations, their is enough latency to move on to Spark.
2. MapReduce technology do not support enough for iterative algorithms if we want to perform and also struggles in stream processing lacking the continous, real-time data processing.
3. It may also need complex programming knowledge in Java which itself makes it a better choice for Python programmers.
4. Apache Spark helps by caching (RAM), providing multi-language API (like PySpark) and libraries like Spark SQL and MLlib.


## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

### Ans: 
Apache Spark does so by following :
1. It uses directly the RAM for gathering data than by doing disk operations(read/write) which is of high latency.
2. Itertive ML algorithms like Logistic Regression and k-means requires the system to make continuous passes over same dataset for updating and optiizing models therefore ultimately needs help from Spark.

## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

### Ans:
Below is the code snippet that will remove all duplicates from the DataFrame based on specific set of columns (user_id and transaction_date) : 

In [ ]:
df_q4_result = average_sales_west(df_sales)
df_q4_result.show()

## Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

### Ans:
We are now Filtering for 'West' region, grouping by product_category, and finding average sale_amount as below:

In [25]:
def average_sales_west(df_sales):
    return (
        df_sales
        .filter(F.col("region") == "West")
        .groupBy("product_category")
        .agg(F.avg("sale_amount").alias("average_sale_amount"))
    )

## Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

### Ans :
.na.drop() completely removes rows from the DataFrame if they contain nulls.
.na.fill() preserves the rows but replaces null values with a specified placeholder.

In [26]:
def fill_null_status(df):
    return df.na.fill({"status": "Unknown"})

## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

### Ans: 

In [27]:
def filter_city_counts(df):
    return (
        df.groupBy("city")
        .count()
        .filter(F.col("count") > 100)
    )

## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

### Ans :
DataFrames cannot be modified in-place. Whenever we drop, rename, or transform a column, Spark creates a new DataFrame reference with an updated execution plan (Lineage). The original DataFrame remains completely untouched in memory, which prevents side effects and allows Spark to optimize operations via a Directed Acyclic Grap (DAG).

## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

### Ans :

In [28]:
def filter_premium_youth(df):
    return df.filter(
        (F.col("age") >= 18) & 
        (F.col("age") <= 30) & 
        (F.col("subscription") == "Premium")
    )

## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

### Ans :
Spark's aggregation functions such as sum() or avg() ignores null values, but if we leave them completely unhandled can cause unexpected results. 
For Example, if an entire column have only null values, then its sum or average will also give, which is not good for downstream steps or dashboards.
Therefore, explicitly handling null values first will guarantee accurate mathematical results. 

## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

### Ans :

In [29]:
def transform_timestamp(df):
    return df.withColumn(
        "event_time", 
        F.col("raw_timestamp").cast(TimestampType())
    ).drop("raw_timestamp")

## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

### Ans: 
Shuffling is the process of redistributing data across different cluster nodes so that rows sharing the same key end up on the exact same physical executor partition. 
It is a wide transformation because it breaks partition boundaries. Computing a single output partition requires reading data from multiple input partitions across the network, resulting in heavy disk and network I/O.

## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

### Ans:

In [1]:
def remove_invalid_users(df):
    return df.filter(
        F.col("email").isNotNull() & 
        (F.col("username") != "")
    )

## Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

## Ans:

In [2]:
def calculate_price_stats(df):
    return df.agg(
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.avg("price").alias("mean_price")
    )

## Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

### Ans:
If dates are inconsistent (e.g., mixing 'YYYY-MM-DD' and 'DD/MM/YYYY'), Spark's schema inference mechanism will fail to match a unique date pattern. Due to which as a safety issue, Spark will infer the entire column as a generic StringType. This forces us to manually parse strings later and risks silently creating null values during subsequent runtime date calculations.

## Q15: Write a final processing pipeline that:
### 1. Filters out duplicates.
### 2. Fills null prices with 0.
### 3. Groups by store_id to calculate total revenue.

### Ans:

In [32]:
def final_processing_pipeline(df_input):
    # 1. Removing duplicate rows
    df_deduplicated = df_input.dropDuplicates()
    
    # 2. Filling null values in the price column with 0
    df_filled = df_deduplicated.na.fill({"price": 0})
    
    # 3. Group by store_id and calculating total revenue
    df_output = (
        df_filled
        .groupBy("store_id")
        .agg(F.sum("price").alias("total_revenue"))
    )
    
    return df_output

# Brief Insights:

### Deduplication Phase:
 The initial dropDuplicates() step removed the identical $ 300.0 entry for user_1, preventing artificial revenue inflation.
### Null Imputation: 
 The na.fill({"price": 0}) command safely converted the missing value for user_4 to dollar 0 instead of dropping the row, keeping store tracking counts intact.
### Business Optimization: 
 Combining data hygiene steps with a groupBy aggregation directly outputs clean, production-ready metrics regarding individual store performance (store_A leading with dollar 500.0).